# Brent · `street_name` (web: "Street name")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `street_name` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "street_name"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "street_name"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["street_name"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,33324,90.18
empty or blank,5,0.01
filled,3623,9.80
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 33,324 | 90.18% |
| empty or blank | 5 | 0.01% |
| filled | 3,623 | 9.80% |

- **Mostly empty:** only 9.80% of records have a `street_name`.
- **5 are blank** (empty or whitespace only).

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["street_name"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["street_name"].value_counts().head(30).to_frame("count")

,count
street_name,
NoneType,33324
str,3628


values that are not strings: 0


,count
street_name,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 3,628 |
| null | 33,324 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Most common values

Count identical values of `street_name` in `df` (exact text) and show the 30 most common.
`shown_as` uses `repr()` so hidden characters are visible.

In [5]:
print("distinct values:", df["street_name"].nunique(), "| filled:", df["street_name"].notna().sum())
top = df["street_name"].value_counts().head(30).to_frame("count").reset_index()
top["shown_as"] = top["street_name"].map(repr)
top

distinct values: 967 | filled: 3628


,street_name,count,shown_as
0,High Road,126,'High Road'
1,Harrow Road,81,'Harrow Road'
2,Ealing Road,47,'Ealing Road'
3,Kilburn High Road,45,'Kilburn High Road'
4,Wembley Hill Road,41,'Wembley Hill Road'
5,High Street,40,'High Street'
6,Cricklewood Broadway,37,'Cricklewood Broadway'
7,Craven Park Road,35,'Craven Park Road'
8,Willesden Lane,33,'Willesden Lane'
9,Chamberlayne Road,30,'Chamberlayne Road'


### Findings · Step 3

Live API, 2026-11-10: 3,628 filled, 967 distinct.

- **Real street names**, led by Brent's main roads: High Road (126), Harrow Road (81),
  Ealing Road (47), Kilburn High Road (45).
- **Case varies:** `North Circular Road` (28) and `NORTH CIRCULAR ROAD` (19) are the same
  street; `EMPIRE WAY` and `CUMBERLAND AVENUE` are in upper case.
- `High Road` and `High Street` are generic names that exist in several parts of Brent;
  on their own they don't say where.